In [1]:
import pandas as pd
import os
import pickle
import bmra_prep
import bmra_prep.pathway_activity.prediction

In [2]:
cell_line ='BC3C'

data_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}/00_outputs_2020_{cell_line}/"
out_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}/01_outputs_2020_{cell_line}/"


os.makedirs(out_dir, exist_ok = True)

# Load Data

In [3]:
# load metdadata dict and extract used elements
with open(os.path.join(data_dir, "metadata.pickle"), "rb") as f:
    all_metadata = pickle.load(f)

n_modules = all_metadata["n_modules"]
n_genes = all_metadata["n_genes"]
n_experiments = all_metadata["n_experiments"]

modules = all_metadata["modules"]
exp_ids = all_metadata["exp_ids"]
genes = all_metadata["genes"]

In [4]:
# load data
L1000_df = pd.read_csv(
    os.path.join(data_dir, "L1000_Data_norm_data.csv"),
    index_col = 0,
)

x = L1000_df.values
x.shape

(978, 86)

In [5]:
# load doses and perturbation matrix
inhib_conc_matrix = pd.read_csv(
    os.path.join(data_dir, "inhib_conc_annotated.csv"),
    index_col = 0,
).values

ic50_matrix = pd.read_csv(
    os.path.join(data_dir, "ic50_annotated.csv"),
    index_col = 0,
).values

# gamma_matrix = pd.read_csv(
#     os.path.join(data_dir, "gamma_annotated.csv"),
#     index_col = 0,
# ).values

pert_matrix = pd.read_csv(
    os.path.join(data_dir, "pert_annotated.csv"),
    index_col = 0,
).values

In [6]:
# y_true = (1 + gamma_matrix * inhib_conc_matrix / ic50_matrix) / (1 + inhib_conc_matrix / ic50_matrix)

y_true = 1 / (1 + inhib_conc_matrix / ic50_matrix)

display(y_true.shape)
y_true

(10, 86)

array([[1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.66666667e-01,
        6.43086817e-01, 9.43396226e-01, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 1.76817289e-02,
        1.39534884e-01, 6.00000000e-01, 1.00000000e+00, 1.00000000e+00,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00, 2.69272963e-03,
        2.37467018e-02, 1.83673469e-01, 1.00000000e+00, 1.000000

## Run models

In [7]:
a_coeffs = bmra_prep.pathway_activity.prediction.predict_coeffs(
    x, y_true, pert_matrix, 200_000, 10, 10, 10, 100)

In [8]:
a_coeffs_df = pd.DataFrame(a_coeffs, index = modules, columns = genes)
a_coeffs_df.to_csv(os.path.join(out_dir, "a_coeffs.csv"))
#a_coeffs_df = pd.read_csv(os.path.join(out_dir,'a_coeffs.csv'),index_col=0)
#a_coeffs = a_coeffs_df.values
display(a_coeffs_df.astype(bool).sum(axis='columns'))
display(a_coeffs_df)

CDK1_2      978
CDK4_6      978
EGFR        978
Estrogen    978
FGFR        978
PI3K        978
p53         978
TOP2A       978
Src         978
SMAD3       978
dtype: int64

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
CDK1_2,0.000024,-6.066002e-06,1.773925e-05,0.000003,-2.525526e-05,0.000021,0.000005,-0.000004,5.078643e-06,-3.618073e-05,...,0.000006,-1.339884e-05,0.000014,-0.000006,-0.000004,0.000035,1.747678e-05,-8.967173e-06,-0.000024,-6.636271e-06
CDK4_6,0.000001,-1.918928e-05,1.202314e-05,0.000010,-2.820385e-06,0.000006,-0.000007,0.000019,-2.173122e-06,-1.444869e-05,...,0.000010,-1.784978e-07,-0.000025,0.000010,-0.000009,-0.000034,-2.461054e-05,-1.561179e-05,-0.000003,1.073767e-05
EGFR,-0.000058,-8.713887e-06,9.773588e-06,-0.000005,-2.170542e-05,-0.000285,0.000005,-0.000018,3.218005e-06,-2.693726e-02,...,-0.000002,-8.344450e-06,-0.000005,-0.000013,-0.000012,0.000297,7.566262e-07,3.413874e-05,0.000022,-3.827658e-06
Estrogen,0.000006,3.264153e-06,1.876193e-05,0.000005,1.842833e-05,-0.000005,-0.000019,0.000015,5.772655e-05,-2.739980e-01,...,0.000020,-1.605948e-06,0.000005,0.000006,-0.000002,-0.000006,-4.913936e-06,-4.725870e-07,-0.000041,1.890196e-05
FGFR,-0.000377,-3.590373e-09,-6.720770e-06,0.000018,2.171661e-05,0.000003,0.000001,0.000011,-4.452782e-06,-7.850532e-05,...,-0.000008,2.854143e-05,0.000016,-0.000003,0.000023,0.000085,-1.142137e-05,3.713088e-05,-0.000009,1.749796e-05
PI3K,0.000031,8.611863e-07,1.768433e-05,-0.000007,4.116176e-06,-0.000023,0.000011,0.000019,1.793166e-05,-2.614474e-06,...,0.000025,-1.758926e-05,0.000021,0.000004,0.000029,-0.000011,5.545090e-06,-1.604153e-05,0.000002,2.058581e-07
p53,0.000016,-2.178199e-05,1.144987e-05,0.000011,-5.352830e-06,-0.000014,0.245878,-0.000011,1.867238e-05,-1.093979e-05,...,0.000021,2.878124e-06,-0.000015,-0.000002,0.000027,0.000002,1.570335e-05,-4.130331e-06,0.000008,4.359702e-05
TOP2A,0.000023,-3.154164e-05,-1.804809e-05,-0.000005,4.328349e-05,-0.000023,-0.000029,0.000018,7.524091e-06,4.863023e-05,...,-0.000038,1.993521e-05,-0.000022,-0.000024,0.000030,0.000047,4.180666e-05,1.036878e-05,-0.000016,1.241005e-05
Src,-0.000012,-4.722082e-06,-5.411000e-07,-0.000026,-1.879172e-07,0.000019,0.000002,-0.000003,9.717583e-07,-4.490026e-05,...,-0.000004,8.370774e-06,0.000004,0.000035,0.000005,-0.000045,-1.830503e-06,2.179686e-05,-0.000006,-8.845737e-06
SMAD3,-0.000021,1.434471e-05,-4.037915e-06,0.000007,-9.126096e-07,0.000028,0.000020,-0.000015,-8.758932e-06,9.025379e-07,...,0.000014,5.070049e-06,-0.000006,0.000025,-0.000006,0.000005,1.240088e-05,1.877910e-05,-0.000007,-5.111359e-07


In [9]:
#pathway_activity = a_coeffs @ x
#pathway_activity.shape

In [9]:
R_global = bmra_prep.pathway_activity.calc_global_response_from_pathway_activity(
    bmra_prep.pathway_activity.calc_pathway_activity(x,a_coeffs),
    modules,
    L1000_df.columns
)
R_global_df = R_global.dataframe
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:A15,MOAR010_BC3C_24H:J22,MOAR010_BC3C_24H:J23,MOAR010_BC3C_24H:J24,MOAR010_BC3C_24H:K07,MOAR010_BC3C_24H:K08,MOAR010_BC3C_24H:K09,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09
CDK1_2,-0.820160,-0.642862,0.065362,-0.087737,0.093715,-0.286267,0.001931,-0.276689,0.119766,0.047070,...,0.064573,0.059483,-0.001462,-0.229014,-0.221563,-0.101776,-0.009756,-0.845880,0.105543,-0.064859
CDK4_6,-0.024899,-0.234373,-0.042053,-0.515399,0.040071,0.002284,-0.007493,0.097552,0.085928,0.031881,...,-0.026570,-0.082978,0.190277,-0.098058,0.468754,0.097383,0.188448,0.318053,0.198911,-0.063431
EGFR,0.593692,0.494285,0.227224,0.364974,0.488497,0.111969,-0.411703,0.262709,0.284063,-0.047412,...,-0.337399,-0.257564,-0.179870,-0.336364,0.145640,-0.044827,0.178441,-0.445349,-0.553711,-0.313522
Estrogen,-0.127860,-0.214846,-0.213716,-0.410911,-0.953006,-0.307857,-0.087312,-0.243954,-0.168179,-0.042541,...,0.040748,0.136420,-0.120636,-0.103847,0.133259,-0.113612,0.108379,-1.526867,-0.225656,0.184068
FGFR,-0.105079,-0.188275,-0.088864,0.053075,-0.031325,-0.400487,-0.025607,-0.059265,-0.071358,-0.285025,...,0.178922,0.199166,0.230097,-0.858495,-0.763273,-0.646301,0.266080,-0.530704,0.053557,0.236427
PI3K,-1.925800,-1.735676,-1.502875,-1.303517,-0.688591,-0.194374,-0.028099,-0.490470,-1.181865,-0.218666,...,0.545062,-0.046428,0.152776,0.440799,-0.262550,0.225010,-0.224851,-1.082295,0.362620,-0.038495
p53,-0.269218,-0.266613,-0.164467,-0.410371,0.029057,-1.627852,-1.476743,-0.114234,-0.095967,-1.327216,...,-0.151053,-0.118611,-0.161732,-0.080834,-1.825272,-1.507098,-1.787053,-0.221455,-0.050382,0.031325
TOP2A,-0.222953,0.075958,-0.231133,-0.172164,-0.125739,0.051805,0.068254,-1.956769,-0.204447,-0.289599,...,-0.036545,-0.198105,-0.195023,-0.016458,-0.145712,0.236897,0.169416,-0.744470,-0.008686,-0.030539
Src,-0.925197,-1.670718,0.525593,-1.210373,0.573652,-1.121370,0.496075,0.399822,0.397925,0.449145,...,0.158544,0.147088,0.013150,0.038902,0.185428,0.314120,0.080716,0.034128,0.242683,0.776194
SMAD3,-0.000005,0.000150,0.000287,0.000293,0.000073,0.000340,0.000037,0.000125,0.000200,0.000251,...,-0.006421,-0.000401,0.000287,-0.000014,0.000092,0.000072,0.000239,0.000294,0.000265,0.000555


In [10]:
R_global_df.to_csv(os.path.join(out_dir, "R_global_annotated.csv"))
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:A15,MOAR010_BC3C_24H:J22,MOAR010_BC3C_24H:J23,MOAR010_BC3C_24H:J24,MOAR010_BC3C_24H:K07,MOAR010_BC3C_24H:K08,MOAR010_BC3C_24H:K09,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09
CDK1_2,-0.820160,-0.642862,0.065362,-0.087737,0.093715,-0.286267,0.001931,-0.276689,0.119766,0.047070,...,0.064573,0.059483,-0.001462,-0.229014,-0.221563,-0.101776,-0.009756,-0.845880,0.105543,-0.064859
CDK4_6,-0.024899,-0.234373,-0.042053,-0.515399,0.040071,0.002284,-0.007493,0.097552,0.085928,0.031881,...,-0.026570,-0.082978,0.190277,-0.098058,0.468754,0.097383,0.188448,0.318053,0.198911,-0.063431
EGFR,0.593692,0.494285,0.227224,0.364974,0.488497,0.111969,-0.411703,0.262709,0.284063,-0.047412,...,-0.337399,-0.257564,-0.179870,-0.336364,0.145640,-0.044827,0.178441,-0.445349,-0.553711,-0.313522
Estrogen,-0.127860,-0.214846,-0.213716,-0.410911,-0.953006,-0.307857,-0.087312,-0.243954,-0.168179,-0.042541,...,0.040748,0.136420,-0.120636,-0.103847,0.133259,-0.113612,0.108379,-1.526867,-0.225656,0.184068
FGFR,-0.105079,-0.188275,-0.088864,0.053075,-0.031325,-0.400487,-0.025607,-0.059265,-0.071358,-0.285025,...,0.178922,0.199166,0.230097,-0.858495,-0.763273,-0.646301,0.266080,-0.530704,0.053557,0.236427
PI3K,-1.925800,-1.735676,-1.502875,-1.303517,-0.688591,-0.194374,-0.028099,-0.490470,-1.181865,-0.218666,...,0.545062,-0.046428,0.152776,0.440799,-0.262550,0.225010,-0.224851,-1.082295,0.362620,-0.038495
p53,-0.269218,-0.266613,-0.164467,-0.410371,0.029057,-1.627852,-1.476743,-0.114234,-0.095967,-1.327216,...,-0.151053,-0.118611,-0.161732,-0.080834,-1.825272,-1.507098,-1.787053,-0.221455,-0.050382,0.031325
TOP2A,-0.222953,0.075958,-0.231133,-0.172164,-0.125739,0.051805,0.068254,-1.956769,-0.204447,-0.289599,...,-0.036545,-0.198105,-0.195023,-0.016458,-0.145712,0.236897,0.169416,-0.744470,-0.008686,-0.030539
Src,-0.925197,-1.670718,0.525593,-1.210373,0.573652,-1.121370,0.496075,0.399822,0.397925,0.449145,...,0.158544,0.147088,0.013150,0.038902,0.185428,0.314120,0.080716,0.034128,0.242683,0.776194
SMAD3,-0.000005,0.000150,0.000287,0.000293,0.000073,0.000340,0.000037,0.000125,0.000200,0.000251,...,-0.006421,-0.000401,0.000287,-0.000014,0.000092,0.000072,0.000239,0.000294,0.000265,0.000555
